In [37]:
import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx

print(jax.__version__)

0.9.1


In [38]:
# initializing arrays
data = [[1, 2, 3], [4, 5, 6]]
arr1 = jnp.array(data)
print(arr1, arr1.dtype, arr1.shape)

[[1 2 3]
 [4 5 6]] int32 (2, 3)


In [39]:
# from NumPy
np_arr = np.array(data)
arr2 = jnp.array(np_arr)
print(arr2, arr2.dtype, arr2.shape)

[[1 2 3]
 [4 5 6]] int32 (2, 3)


In [40]:
ones = jnp.ones_like(arr2)
print(ones, ones.shape, ones.dtype)
zeros = jnp.zeros_like(arr2)
print(zeros, zeros.shape, zeros.dtype)

[[1 1 1]
 [1 1 1]] (2, 3) int32
[[0 0 0]
 [0 0 0]] (2, 3) int32


In [41]:
# constants or random values
shape = (3, 3)
ones = jnp.ones(shape)
zeros = jnp.zeros(shape)

seed = 123
key = jax.random.key(seed)
rand = jax.random.uniform(key, shape)
randn = jax.random.normal(key, shape)

print(f"Ones: {ones}")
print(f"Zeros: {zeros}")
print(f"Uniform Random: {rand}")
print(f"Gaussian Random: {randn}")

Ones: [[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
Zeros: [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Uniform Random: [[0.9490746  0.7997726  0.5088254 ]
 [0.2983067  0.24615324 0.13832462]
 [0.10386252 0.9456022  0.17012143]]
Gaussian Random: [[ 1.6359469   0.8408094   0.02212393]
 [-0.52927685 -0.6866449  -1.0878773 ]
 [-1.2598456   1.6036302  -0.9536853 ]]


In [42]:
randn_2 = jax.random.normal(key, shape)
print((randn == randn_2).all())

True


In [43]:
print(key)

Array((), dtype=key<fry>) overlaying:
[  0 123]


In [44]:
k1, k2 = jax.random.split(key, 2)
print(k1)
print(k2)

Array((), dtype=key<fry>) overlaying:
[ 969380169 3408994049]
Array((), dtype=key<fry>) overlaying:
[2095235069 2959038751]


In [45]:
randn_3 = jax.random.normal(k1, shape)
randn_4 = jax.random.normal(k2, shape)
print((randn_3 != randn_4).all())

True


In [46]:
# initialize from pytorch
import torch

x_torch = torch.randn(shape)
x_jax = jnp.asarray(x_torch)
print(x_jax, x_jax.shape, x_jax.dtype)

[[-0.8858983  -0.25235555 -0.4843839 ]
 [-0.19742131  0.7646769  -0.69853556]
 [ 0.0597363   0.18136665 -1.3221418 ]] (3, 3) float32


In [47]:
# without making a copy
x_jax = jax.dlpack.from_dlpack(x_torch, copy=False)
print(x_jax, x_jax.shape, x_jax.dtype)

[[-0.8858983  -0.25235555 -0.4843839 ]
 [-0.19742131  0.7646769  -0.69853556]
 [ 0.0597363   0.18136665 -1.3221418 ]] (3, 3) float32


In [48]:
print(f"Device: {x_jax.device}")

Device: TFRT_CPU_0


In [49]:
try:
    x_jax[0, 0] = 100.0
except TypeError as e:
    print(e)


x_torch = torch.tensor([1, 2, 3, 4])
x_jax = jnp.array([1, 2, 3, 4])
print(f"Default integer dtypes, PyTorch: {x_torch.dtype} and Jax: {x_jax.dtype}")

x_torch = torch.zeros(3, 4)
x_jax = jnp.zeros((3, 4))
print(f"Default float dtypes, PyTorch: {x_torch.dtype} and Jax: {x_jax.dtype}")
print(f"Default devices, PyTorch: {x_torch.device} and Jax: {x_jax.device}")

JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html
Default integer dtypes, PyTorch: torch.int64 and Jax: int32
Default float dtypes, PyTorch: torch.float32 and Jax: float32
Default devices, PyTorch: cpu and Jax: TFRT_CPU_0


In [50]:
# device accelerator stuff
print(f"Available devices {jax.devices('cpu')}")
x_cpu = jax.device_put(x_jax, device=jax.devices("cpu")[0])

Available devices [CpuDevice(id=0)]


In [51]:
print(x_cpu.device)

TFRT_CPU_0


In [52]:
x = jax.random.normal(key, (3, 4))
print(x)

[[ 1.6359469   0.8408094   0.02212393 -0.52927685]
 [-0.6866449  -1.0878773  -1.2598456   1.6036302 ]
 [-0.9536853  -0.405823   -0.21461229 -1.6922154 ]]


In [53]:
print(x[:, 0])
print(x[0, :])
print(x[..., 0])

[ 1.6359469 -0.6866449 -0.9536853]
[ 1.6359469   0.8408094   0.02212393 -0.52927685]
[ 1.6359469 -0.6866449 -0.9536853]


In [54]:
x = x.at[0, 0].set(0)
print(x)

[[ 0.          0.8408094   0.02212393 -0.52927685]
 [-0.6866449  -1.0878773  -1.2598456   1.6036302 ]
 [-0.9536853  -0.405823   -0.21461229 -1.6922154 ]]


In [55]:
print(jnp.arange(10)[-1])

9


In [56]:
cat = jnp.concat([x, x], axis=-1)
print(cat)

[[ 0.          0.8408094   0.02212393 -0.52927685  0.          0.8408094
   0.02212393 -0.52927685]
 [-0.6866449  -1.0878773  -1.2598456   1.6036302  -0.6866449  -1.0878773
  -1.2598456   1.6036302 ]
 [-0.9536853  -0.405823   -0.21461229 -1.6922154  -0.9536853  -0.405823
  -0.21461229 -1.6922154 ]]


In [57]:
y1 = x @ x.T
y2 = jnp.matmul(x, x.T)
print((y1 == y2).all())

True


In [58]:
print(y1.shape)

(3, 3)


In [59]:
y1 = x * x
y2 = jnp.multiply(x, x)
print((y1 == y2).all())

True


In [60]:
print(y1.shape)

(3, 4)


In [61]:
agg = x.sum()
print(type(agg.item()))

<class 'float'>


In [62]:
# learning autodiff

# Input tensor
x = jax.random.normal(k2, (5))
# Target output
y_true = jax.random.normal(k1, (3))

# Initialize random parameters
seed = 123
key = jax.random.key(seed)
key, w_key, b_key = jax.random.split(key, 3)
w = jax.random.normal(w_key, (5, 3))
b = jax.random.normal(b_key, (3,))


# model function
def predict(x, w, b):
    return jnp.matmul(x, w) + b


# Criterion or loss function
def compute_loss(w, b, x, y_true):
    y_pred = predict(x, w, b)
    return jnp.mean((y_true - y_pred) ** 2)


loss = compute_loss(w, b, x, y_true)
print(loss)

2.425662


In [63]:
loss, (w_grad, b_grad) = jax.value_and_grad(compute_loss, argnums=(0, 1))(
    w, b, x, y_true
)
print(f"{w_grad=}")
print(f"{b_grad=}")
print(f"{loss=}")

w_grad=Array([[-1.1325147 , -0.3203278 ,  0.7724488 ],
       [-2.2241256 , -0.6290861 ,  1.5169984 ],
       [-0.7975925 , -0.22559622,  0.54401   ],
       [-0.03450844, -0.00976059,  0.023537  ],
       [ 1.6844277 ,  0.4764344 , -1.1488893 ]], dtype=float32)
b_grad=Array([ 1.4467386 ,  0.4092049 , -0.98676986], dtype=float32)
loss=Array(2.425662, dtype=float32)


In [64]:
net_params = {"weights": w, "bias": b}


# Criterion or loss function
def compute_loss_2(param_dict, x, y_true):
    y_pred = predict(x, param_dict["weights"], param_dict["bias"])
    return jnp.mean((y_true - y_pred) ** 2)


loss, grad_dict = jax.value_and_grad(compute_loss_2, argnums=0)(net_params, x, y_true)
print(f"{loss=}")
print(f"{grad_dict=}")

loss=Array(2.425662, dtype=float32)
grad_dict={'bias': Array([ 1.4467386 ,  0.4092049 , -0.98676986], dtype=float32), 'weights': Array([[-1.1325147 , -0.3203278 ,  0.7724488 ],
       [-2.2241256 , -0.6290861 ,  1.5169984 ],
       [-0.7975925 , -0.22559622,  0.54401   ],
       [-0.03450844, -0.00976059,  0.023537  ],
       [ 1.6844277 ,  0.4764344 , -1.1488893 ]], dtype=float32)}


In [65]:
# To install Flax: `pip install -U flax treescope optax`


class BasicBlock(nnx.Module):
    def __init__(
        self,
        in_planes: int,
        out_planes: int,
        do_downsample: bool = False,
        *,
        rngs: nnx.Rngs,
    ):
        strides = (2, 2) if do_downsample else (1, 1)
        self.conv1_bn1 = nnx.Sequential(
            nnx.Conv(
                in_planes,
                out_planes,
                kernel_size=(3, 3),
                strides=strides,
                padding="SAME",
                use_bias=False,
                rngs=rngs,
            ),
            nnx.BatchNorm(out_planes, momentum=0.9, epsilon=1e-5, rngs=rngs),
        )
        self.conv2_bn2 = nnx.Sequential(
            nnx.Conv(
                out_planes,
                out_planes,
                kernel_size=(3, 3),
                strides=(1, 1),
                padding="SAME",
                use_bias=False,
                rngs=rngs,
            ),
            nnx.BatchNorm(out_planes, momentum=0.9, epsilon=1e-5, rngs=rngs),
        )

        if do_downsample:
            self.conv3_bn3 = nnx.Sequential(
                nnx.Conv(
                    in_planes,
                    out_planes,
                    kernel_size=(1, 1),
                    strides=(2, 2),
                    padding="VALID",
                    use_bias=False,
                    rngs=rngs,
                ),
                nnx.BatchNorm(out_planes, momentum=0.9, epsilon=1e-5, rngs=rngs),
            )
        else:
            self.conv3_bn3 = lambda x: x

    def __call__(self, x: jax.Array):
        out = self.conv1_bn1(x)
        out = nnx.relu(out)

        out = self.conv2_bn2(out)
        out = nnx.relu(out)

        shortcut = self.conv3_bn3(x)
        out += shortcut
        out = nnx.relu(out)
        return out


class ResNet18(nnx.Module):
    def __init__(self, num_classes: int, *, rngs: nnx.Rngs):
        self.num_classes = num_classes
        self.conv1_bn1 = nnx.Sequential(
            nnx.Conv(
                3,
                64,
                kernel_size=(3, 3),
                strides=(1, 1),
                padding="SAME",
                use_bias=False,
                rngs=rngs,
            ),
            nnx.BatchNorm(64, momentum=0.9, epsilon=1e-5, rngs=rngs),
        )
        self.layer1 = nnx.Sequential(
            BasicBlock(64, 64, rngs=rngs),
            BasicBlock(64, 64, rngs=rngs),
        )
        self.layer2 = nnx.Sequential(
            BasicBlock(64, 128, do_downsample=True, rngs=rngs),
            BasicBlock(128, 128, rngs=rngs),
        )
        self.layer3 = nnx.Sequential(
            BasicBlock(128, 256, do_downsample=True, rngs=rngs),
            BasicBlock(256, 256, rngs=rngs),
        )
        self.layer4 = nnx.Sequential(
            BasicBlock(256, 512, do_downsample=True, rngs=rngs),
            BasicBlock(512, 512, rngs=rngs),
        )
        self.fc = nnx.Linear(512, self.num_classes, rngs=rngs)

    def __call__(self, x: jax.Array):
        x = self.conv1_bn1(x)
        x = nnx.relu(x)
        x = nnx.max_pool(x, (2, 2), strides=(2, 2))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = nnx.avg_pool(x, (x.shape[1], x.shape[2]))
        x = x.reshape((x.shape[0], -1))
        x = self.fc(x)
        return x

In [66]:
model = ResNet18(10, rngs=nnx.Rngs(0))

# Visualize the model architecture
nnx.display(model)

In [67]:
x = jnp.ones((4, 32, 32, 3))
y_pred = model(x)
y_pred.shape

(4, 10)

In [68]:
# CIFAR10 training/testing datasets setup

from torchvision.datasets import CIFAR10
from torchvision.transforms import v2 as T


def to_np_array(pil_image):
    return np.asarray(pil_image)


def normalize(image):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = image.astype(np.float32) / 255.0
    return (image - mean) / std


train_transforms = T.Compose(
    [
        T.Pad(4),
        T.RandomCrop(32, fill=128),
        T.RandomHorizontalFlip(),
        T.Lambda(to_np_array),
        T.Lambda(normalize),
    ]
)

test_transforms = T.Compose(
    [
        T.Lambda(to_np_array),
        T.Lambda(normalize),
    ]
)

train_dataset = CIFAR10("./data", train=True, download=True, transform=train_transforms)
test_dataset = CIFAR10("./data", train=True, download=False, transform=test_transforms)

In [72]:
# Data loaders setup
from torch.utils.data import DataLoader

batch_size = 512


def np_arrays_collate_fn(list_of_datapoints):
    list_of_images = [dp[0] for dp in list_of_datapoints]
    list_of_targets = [dp[1] for dp in list_of_datapoints]
    return np.stack(list_of_images, axis=0), np.asarray(list_of_targets)


train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    num_workers=0,
    shuffle=True,
    collate_fn=np_arrays_collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    num_workers=0,
    shuffle=False,
    collate_fn=np_arrays_collate_fn,
)

In [73]:
# Let us check training dataloader:
trl_iter = iter(train_loader)
batch = next(trl_iter)
print(batch[0].shape, batch[0].dtype, batch[1].shape, batch[1].dtype)

(512, 32, 32, 3) float64 (512,) int64


In [74]:
import optax

learning_rate = 0.005
momentum = 0.9

optimizer = nnx.ModelAndOptimizer(model, optax.adamw(learning_rate, momentum))

In [ ]:
def compute_loss_and_logits(model: nnx.Module, batch):
    logits = model(batch[0])
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=batch[1]
    ).mean()
    return loss, logits


@nnx.jit
def train_step(
    model: nnx.Module, optimizer: nnx.Optimizer, metrics: nnx.MultiMetric, batch
):
    """Train for a single step."""
    # convert numpy arrays to jnp.array on GPU
    x, y_true = jnp.asarray(batch[0]), jnp.asarray(batch[1])

    grad_fn = nnx.value_and_grad(compute_loss_and_logits, has_aux=True)
    (loss, logits), grads = grad_fn(model, (x, y_true))

    metrics.update(loss=loss, logits=logits, labels=y_true)  # In-place updates.

    optimizer.update(grads)  # In-place updates.
    return loss


@nnx.jit
def eval_step(model: nnx.Module, metrics: nnx.MultiMetric, batch):
    # convert numpy arrays to jnp.array on GPU
    x, y_true = jnp.asarray(batch[0]), jnp.asarray(batch[1])

    loss, logits = compute_loss_and_logits(model, (x, y_true))

    metrics.update(loss=loss, logits=logits, labels=y_true)  # In-place updates.

In [76]:
# Define helper object to compute train/test metrics
metrics = nnx.MultiMetric(
    accuracy=nnx.metrics.Accuracy(),
    loss=nnx.metrics.Average("loss"),
)

metrics_history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": [],
}

In [ ]:
# Start the training

num_epochs = 3

for epoch in range(num_epochs):
    metrics.reset()  # Reset the metrics for the test set.

    model.train()  # Set model to the training mode: e.g. update batch statistics
    for step, batch in enumerate(train_loader):
        loss = train_step(model, optimizer, metrics, batch)

        print(
            f"\r[train] epoch: {epoch + 1}/{num_epochs}, iteration: {step}, batch loss: {loss.item():.4f}",
            end="",
        )
    print("\r", end="")

    for metric, value in metrics.compute().items():  # Compute the metrics.
        metrics_history[f"train_{metric}"].append(value)  # Record the metrics.
    metrics.reset()  # Reset the metrics for the test set.

    # Compute the metrics on the test set after each training epoch.
    model.eval()  # Set model to evaluation model: e.g. use stored batch statistics
    for test_batch in test_loader:
        eval_step(model, metrics, test_batch)

    # Log the test metrics.
    for metric, value in metrics.compute().items():
        metrics_history[f"test_{metric}"].append(value)
    metrics.reset()  # Reset the metrics for the next training epoch.

    print(
        f"[train] epoch: {epoch + 1}/{num_epochs}, "
        f"loss: {metrics_history['train_loss'][-1]:0.4f}, "
        f" accuracy: {metrics_history['train_accuracy'][-1] * 100:0.4f}"
    )
    print(
        f"[test] epoch: {epoch + 1}/{num_epochs}, "
        f"loss: {metrics_history['test_loss'][-1]:0.4f}, "
        f"accuracy: {metrics_history['test_accuracy'][-1] * 100:0.4f}"
        "\n"
    )

In [78]:
def matmul_relu_add(x, y):
    z = x * y
    return jax.nn.relu(z) + x

In [79]:
key = jax.random.key(123)
key1, key2 = jax.random.split(key)
x = jax.random.uniform(key1, (2500, 3000))
y = jax.random.uniform(key2, (2500, 3000))

In [80]:
%%timeit
matmul_relu_add(x, y)

6.07 ms ± 447 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [81]:
jit_matmul_relu = jax.jit(matmul_relu_add)
# Warm-up: compile the function
_ = jit_matmul_relu(x, y)

In [82]:
%%timeit
jit_matmul_relu(x, y)

1.72 ms ± 22.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
